In [ ]:
import os
os.chdir('/workspaces/korea-real-estate-population-movement')

from src.config import DB_URL
import pandas as pd
from sqlalchemy import create_engine, inspect
import matplotlib.pyplot as plt

In [ ]:
# Add Korean font
import matplotlib.font_manager as fm
fm.fontManager.addfont(
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
)
[f.name for f in fm.fontManager.ttflist if "Nanum" in f.name]
plt.rcParams["font.family"] = "NanumGothic"

In [ ]:
engine = create_engine(DB_URL)

# Helper function to read sql file
def read_sql_file(path):
    with open(path,"r") as f:
        return f.read()


In [ ]:
# Inspect tables before calling data

inspector = inspect(engine)

for table in inspector.get_table_names():
    print(f"\n--- {table} ---")
    
    for column in inspector.get_columns(table):
        print(column["name"], column["type"])

In [ ]:
df = pd.read_sql(
    f"SELECT * FROM seoul_population_flow LIMIT 5",
    engine
)

display(df)


# Districts with the highest number of inflow and outflow

In [ ]:
cumulative_outflow_query = read_sql_file('/workspaces/korea-real-estate-population-movement/sql/analysis/top_five_cumulative_outflow.sql')
yearly_top_outflow_query = read_sql_file('/workspaces/korea-real-estate-population-movement/sql/analysis/yearly_top_outflow.sql')

top_outflow_districts = pd.read_sql_query(cumulative_outflow_query,engine)
yearly_outflow = pd.read_sql_query(yearly_top_outflow_query,engine)

display(top_outflow_districts)
display(yearly_outflow)

In [ ]:
cumulative_inflow_query = read_sql_file('/workspaces/korea-real-estate-population-movement/sql/analysis/top_five_cumulative_inflow.sql')
yearly_top_inflow_query = read_sql_file('/workspaces/korea-real-estate-population-movement/sql/analysis/yearly_top_inflow.sql')

top_inflow_districts = pd.read_sql_query(cumulative_inflow_query,engine)
yearly_inflow = pd.read_sql_query(yearly_top_inflow_query,engine)

display(top_inflow_districts)
display(yearly_inflow)

## Observation

- After identifying districts with high inflow and outflow, we have decided to focus on the 7 districts that fall in that category: Gangnam, Songpa, Gangdong, Seocho, Gwanak, Dongjak, Yeongdeungpo. 

- Next, we are going to query the turnover rate of each of these districts over the last three years. 
    - Turnover = Inflow + Outflow

In [ ]:
yearly_turnover_query = read_sql_file('/workspaces/korea-real-estate-population-movement/sql/analysis/yearly_turnover.sql')
yearly_turnover = pd.read_sql_query(yearly_turnover_query,engine)

display(yearly_turnover)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3,figsize=(14,6))

district_colors = {
    "강남구": "tab:blue",
    "송파구": "tab:orange",
    "강동구": "tab:green",
    "서초구": "tab:red",
    "관악구": "tab:purple",
    "동작구": "tab:olive",
    "영등포구": "tab:pink"
}

for district in top_outflow_districts['from_district']:
    district_df = yearly_outflow[yearly_outflow['from_district'] == district]
    ax1.plot(
        district_df['year'],
        district_df['total_outflow'],
        marker="o",
        label=district,
        color=district_colors[district]
    )

ax1.set_xticks([2023, 2024, 2025])
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Outflow")
ax1.set_title("People Moving Out")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

for district in top_inflow_districts['to_district']:
    district_df = yearly_inflow[yearly_inflow['to_district'] == district]

    ax2.plot(
        district_df['year'],
        district_df['total_inflow'],
        marker='o',
        label=district,
        color=district_colors[district]
    )

ax2.set_xticks([2023, 2024, 2025])
ax2.set_xlabel("Year")
ax2.set_ylabel("Total Inflow")
ax2.set_title("People Moving In")
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

for district in yearly_turnover['district'].unique():
    district_df = yearly_turnover[yearly_turnover['district'] == district]

    ax3.plot(
        district_df['year'],
        district_df['turnover'],
        marker='o',
        label=district,
        color=district_colors[district]
    )

ax3.set_xticks([2023, 2024, 2025])
ax3.set_xlabel('Year')
ax3.set_ylabel('Total Turnover')
ax3.set_title('Total People Moving In & Out')
ax3.legend()
ax3.grid(axis='y',alpha=0.3)

plt.tight_layout()
plt.show()

Outflow & Inflow Observations:

- Gangnam's net population movement has shifted from negative to positive.
    - Outflow: ~41K -> ~39K -> ~32K
    - Inflow: ~31K -> ~32K -> ~34K
- Gangdong has a huge increase in outflow over the three years
    - ~19K -> ~32K -> ~36K
    - Why did Gangdong record such large increases in resident outflow?
- Songpa is gaining population
    - While outflow stayed relatively steady, inflow is showing a increasing trend
        - Inflow: ~30K -> ~33K -> ~35K
- Gwanak's net population movement is becoming increasingly negative
    - Pairing a slow steady increase in outflow and decrease in inflow, Gwanak is slowly decreasing its overall net movement

Turnover Observations:
- Gangnam has the highest turnover with recently also recording a net positive population movement.
    - It has the highest amount of activity amongst the 7 districts.
- Songpa displays an increasing turnover trend over the last three years.
    - Songpa records the highest number of residents moving in while almost closing the gap with Gangnam for the highest turnover.
    - Songpa's population is growing in a city, creating a growing net positive population movement
- Gangdong, however, is the opposite. Although it has a increasing turnover trend in the observation period, Gangdong surpassed as the number 1 spot for highest residents moving out in 2025. 

 

# Individual Apartment Sales Data in Seoul

Individual Apartment Sales data holds every transaction of an apartment sale in Seoul. 

In [ ]:
inspector = inspect(engine)

for column in inspector.get_columns('individual_apt_sales'):
    print(column["name"], column["type"])

In [ ]:
top_transaction_districts_query = read_sql_file('/workspaces/korea-real-estate-population-movement/sql/analysis/top_five_transactions.sql')

top_transaction_districts = pd.read_sql(top_transaction_districts_query,engine)

display(top_transaction_districts)

## Observation on cumulative transaction count from 2023 - 2025

Observation:
- Songpa's interesting relationship
    - Highest apartment transaction volume
    - High Inflow
    - Stable Outflow
    - Increasing Turnover
- Nowon and Sungbook did not appear in our earlier observations. This may be a indication high apartment transaction activity doesn't necessarily correlate to a high population flow pattern.
- Gangdong
    - 3rd largest transaction volume
    - Sharp increase in outflow and turnover
    - Could Gangdong's high population movement be associated with its high housing transaction activity?
- Gangnam
    - Highest turnover amongst our 7 districts
    - Declining outflow
    - Increasing inflow
    - 4th highest transaction volume


In [ ]:
# Transaction for the 5 district for each year

yearly_top_transactions_query = read_sql_file('/workspaces/korea-real-estate-population-movement/sql/analysis/yearly_top_transaction.sql')
yearly_top_transactions = pd.read_sql(yearly_top_transactions_query,engine)

display(yearly_top_transactions)

In [ ]:
district_colors = {
    "강남구": "tab:blue",
    "송파구": "tab:orange",
    "강동구": "tab:green",
    "서초구": "tab:red",
    "관악구": "tab:purple",
    "동작구": "tab:olive",
    "영등포구": "tab:pink",
    "성북구": "tab:brown",
    "노원구": "tab:gray"
}

fig, ax = plt.subplots(figsize=(10,6))

for district in yearly_top_transactions['district'].unique():
    district_df = yearly_top_transactions[yearly_top_transactions['district'] == district]

    ax.plot(
        district_df['deal_year'],
        district_df['transaction_count'],
        marker='o',
        label=district
    )

ax.set_xticks([2023, 2024, 2025])
ax.set_xlabel('Year')
ax.set_ylabel('Transaction Count')
ax.set_title('Apartment Sale Transaction Count')
ax.legend()
ax.grid(axis='y',alpha=0.3)

plt.tight_layout()
plt.show()

## Apartment Sale Transaction Observation

- Every district increased substantially over the 3 years
- Songpa and Nowon had the highest transaction activity with Nowon surpassing Songpa for the top spot.
- Gangdong had the sharpest increase in transaction count
    - Sharp increase in both Population Outflow and Apartment Transactions
- Gangnam by year 2025 had the lowest transaction count out of the 5 districts. 
    - Note: Gangnam had the highest population turnover 

In [19]:
# Combine population flow and sale transaction df

yearly_top_transactions = yearly_top_transactions.rename(columns={"deal_year":"year"})

combined_df = yearly_top_transactions.merge(
    yearly_turnover,
    on=['district','year'],
    how='inner')

for district in combined_df['district'].unique():
    display(combined_df[combined_df['district'] == district])



,district,year,transaction_count,total_outflow,total_inflow,turnover
0,송파구,2023,2757,29824,30054,59878
3,송파구,2024,4357,31433,32235,63668
6,송파구,2025,5640,30204,35199,65403


,district,year,transaction_count,total_outflow,total_inflow,turnover
1,강남구,2023,2339,40941,30791,71732
4,강남구,2024,3754,38734,31058,69792
8,강남구,2025,4318,31804,34147,65951


,district,year,transaction_count,total_outflow,total_inflow,turnover
2,강동구,2023,2195,18988,16513,35501
5,강동구,2024,3442,32250,15898,48148
7,강동구,2025,5513,36447,17466,53913


## Observations

- Songpa experienced increasing apartment transaction activity alongside increasing population inflow and turnover.
- Gangnam, on the other hand, recorded an increasing apartment transaction activity despite decreasing turnover. 
    - This suggest higher transaction activity did not correspond with increased population movement during the observation period.
- Gangdong experienced increasing apartment transaction acitivity alongside population outflow, and turnover. Because Gangdong's population inflow remained relatively steady, Gangdong's increased turnover is primarily driven by outflow. 
    - This raises the question, what could explain why Gangdong between 2023 and 2025 experienced simultaneous increase in apartment transactions and resident outflow?